# Notebook 02: Discrete Dividend Treatment Comparison
Discrete cash dividends cause spot stock prices to drop on ex-dividend dates, breaking the standard GBM lognormality assumption. This notebook compares two popular dividend treatments:
1. **Escrowed Dividend Model**: Subtracts the PV of all future dividends from the spot price: $S_{adj} = S_0 - \sum D_i e^{-r t_i}$.
2. **Piecewise Lognormal Model (Vellekoop-Nieuwenhuis)**: Adjusts binomial tree nodes on ex-dividend dates and interpolates continuation values.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from dpp.core.models import PricingParams
from dpp.core.instruments import DividendSchedule
from dpp.pricers.binomial_tree import BinomialTreeModel

# Option & Dividend parameters
spot = 100.0
strike = 100.0
T = 1.0
r = 0.05
sigma = 0.20

# Dividend schedule: $2.00 paid at t=0.50
div_schedule = DividendSchedule(dividends=[(0.50, 2.00)])
params = PricingParams(spot=spot, strike=strike, maturity=T, rate=r, div_yield=0.0, sigma=sigma, option_type="call")


In [ ]:
# Compare pricing across tree steps
steps_range = np.arange(20, 151, 10)
prices_escrowed = []
prices_pw = []

for steps in steps_range:
    # Escrowed model
    model_escrowed = BinomialTreeModel(n_steps=steps, treatment="escrowed")
    prices_escrowed.append(model_escrowed.price(params, dividend_schedule=div_schedule))
    
    # Piecewise lognormal model
    model_pw = BinomialTreeModel(n_steps=steps, treatment="piecewise_lognormal")
    prices_pw.append(model_pw.price(params, dividend_schedule=div_schedule))

prices_escrowed = np.array(prices_escrowed)
prices_pw = np.array(prices_pw)


In [ ]:
# Plot convergence comparison
plt.figure(figsize=(10, 6))
plt.plot(steps_range, prices_escrowed, marker='o', color='blue', label='Escrowed Dividend Model')
plt.plot(steps_range, prices_pw, marker='s', color='green', label='Piecewise Lognormal Model (VN)')
plt.axhline(prices_pw[-1], color='red', linestyle='--', label='Asymptotic Price')
plt.title("Dividend Treatment Convergence in Binomial Trees")
plt.xlabel("Number of Tree Steps")
plt.ylabel("Call Option Price")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Display price discrepancy details
print(f"Price with Escrowed Model (150 steps):       {prices_escrowed[-1]:.4f}")
print(f"Price with Piecewise Lognormal VN (150 steps): {prices_pw[-1]:.4f}")
print(f"Difference:                                  {np.abs(prices_escrowed[-1] - prices_pw[-1]):.6f}")
